# Notebook 10 – Feature Importance

This notebook covers how to measure which features a model relies on most, and why high importance does not always mean cause and effect.

In [4]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34


**Code Explanation:** The dataset is loaded, rows with missing CustomerID are removed, and a TotalPrice column is created from Quantity and UnitPrice for use in the examples below.

## 1. What is Feature Importance?

**Definition:** A score that shows how much each feature contributed to a model's predictions.

**Example:** In a sales prediction model, UnitPrice might get an importance score of 0.6 while Country gets only 0.05.

**Why it is used?** Helps understand what the model is actually paying attention to, and which features are driving its decisions.

In [11]:
from sklearn.preprocessing import LabelEncoder
df['Country_Code'] = LabelEncoder().fit_transform(df['Country'])
features = df[['Quantity', 'UnitPrice', 'Country_Code']]
target = df['TotalPrice']
features.head()

,Quantity,UnitPrice,Country_Code
0,6,2.55,35
1,6,3.39,35
2,8,2.75,35
3,6,3.39,35
4,6,3.39,35


**Code Explanation:** Country is converted into numeric codes since models need numbers, and the feature set along with the target column is prepared for the models used later.

## 2. Model-Based Feature Importance

**Definition:** Feature importance scores that come directly from a trained model, based on how that model used each feature internally.

**Example:** A Random Forest naturally produces importance scores after training, without any extra steps.

**Why it is used?** Gives a quick, built-in way to rank features without needing separate calculations.

In [12]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(features, target)
model.feature_importances_

array([8.68231210e-01, 1.31641571e-01, 1.27219248e-04])

**Code Explanation:** A Random Forest model is trained on the features, and feature_importances_ returns a score for each feature showing how much it was used across all the trees.

## 3. Decision Tree Feature Importance

**Definition:** Importance scores from a single Decision Tree, based on how much each feature reduced error when splitting the data.

**Example:** If UnitPrice is used for the first split and separates the data well, it gets a high importance score.

**Why it is used?** Simple way to see which features a tree relied on most for its decisions.

In [13]:
from sklearn.tree import DecisionTreeRegressor
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(features, target)
tree_model.feature_importances_

array([8.87585834e-01, 1.12400758e-01, 1.34081700e-05])

**Code Explanation:** A single Decision Tree is trained on the same features, and feature_importances_ shows how much each feature contributed to reducing prediction error across its splits.

## 4. Random Forest Feature Importance

**Definition:** Feature importance averaged across all the trees inside a Random Forest, making it more stable than a single tree.

**Example:** Even if one tree ranks UnitPrice low, the overall forest average may still rank it high if most trees rely on it.

**Why it is used?** More reliable than a single tree since it averages out randomness from individual trees.

In [14]:
importances = pd.Series(model.feature_importances_, index=features.columns)
importances.sort_values(ascending=False)

Quantity        0.868231
UnitPrice       0.131642
Country_Code    0.000127
dtype: float64

**Code Explanation:** The importance scores from the Random Forest are matched with their feature names and sorted from most to least important.

## 5. Permutation Importance

**Definition:** A method that measures importance by shuffling one feature's values and checking how much the model's performance drops.

**Example:** If shuffling UnitPrice badly hurts accuracy, it is an important feature. If shuffling Country barely changes anything, it is not.

**Why it is used?** Works with any model, not just tree-based ones, and directly measures impact on performance rather than internal structure.

In [15]:
from sklearn.inspection import permutation_importance
result = permutation_importance(model, features, target, n_repeats=10, random_state=42)
pd.Series(result.importances_mean, index=features.columns).sort_values(ascending=False)

Quantity        1.689541
UnitPrice       0.185417
Country_Code    0.000048
dtype: float64

**Code Explanation:** permutation_importance shuffles each feature 10 times and measures the drop in model performance each time, then the average drop per feature is displayed as its importance score.

## 6. Feature Importance vs Feature Selection

**Definition:** Feature importance ranks how much each feature matters, while feature selection is the actual process of keeping or removing features based on that ranking or other criteria.

**Example:** Feature importance tells you UnitPrice matters most. Feature selection is the decision to drop Country_Code because its score was too low.

**Why it is used?** Importance is about understanding, selection is about acting on that understanding to build a leaner model.

In [16]:
threshold = 0.05
selected_features = importances[importances > threshold].index.tolist()
selected_features

['Quantity', 'UnitPrice']

**Code Explanation:** A simple threshold is applied to the importance scores, and only features above that threshold are kept as the selected features.

## 7. Interpreting Feature Importance

**Definition:** The process of understanding what a feature's importance score actually means in context, rather than just looking at the number.

**Example:** A high importance score for UnitPrice makes sense since TotalPrice is directly calculated from it.

**Why it is used?** Prevents misreading scores, and connects the numbers back to real-world meaning in the data.

In [17]:
for feature, score in importances.sort_values(ascending=False).items():
    print(feature, round(score, 3))

Quantity 0.868
UnitPrice 0.132
Country_Code 0.0


**Code Explanation:** Each feature and its importance score are printed together in ranked order, making the results easy to read and interpret.

## 8. Limitations of Feature Importance

**Definition:** The known weaknesses and blind spots of feature importance scores that make them unreliable if trusted blindly.

**Example:** Two highly correlated features can split the same importance between them, making both look less important than they really are.

**Why it is used?** Understanding these limitations stops teams from making wrong conclusions based only on importance scores.

In [18]:
df[['Quantity', 'UnitPrice', 'TotalPrice']].corr()

,Quantity,UnitPrice,TotalPrice
Quantity,1.000000,-0.001238,0.916096
UnitPrice,-0.001238,1.000000,-0.129296
TotalPrice,0.916096,-0.129296,1.000000


**Code Explanation:** Correlation between the features is checked, since highly correlated features are a common reason importance scores can be misleading or split unfairly.

## Why High Importance Does Not Mean Causation

A feature getting a high importance score only means the model found it useful for making accurate predictions. It does not mean that feature actually causes the target to change in the real world.

**Reason 1: Correlation is not causation.** UnitPrice may be important for predicting TotalPrice simply because they are mathematically linked, not because UnitPrice is causing anything in a real-world sense.

**Reason 2: Hidden confounders.** A feature might look important only because it is connected to some other hidden factor that is the real driver, like a season or trend that affects both the feature and the target together.

**Reason 3: Data leakage.** Sometimes a feature is important only because it accidentally contains information from the target itself, which will not hold up on new, unseen data.

**Reason 4: Correlated features split credit.** If two features carry similar information, a model might rely on either one, making importance scores unstable and not a clean measure of real-world impact.

Because of this, feature importance should be treated as a signal of usefulness to the model, not as proof of a true cause and effect relationship. Confirming actual causation needs separate methods, like controlled experiments or causal inference techniques.